# SAR Oil Spill — Preprocessing Notebook

Run this **once**. It:
1. Computes normalization statistics (mean/std) from the training set
2. Saves the train/val/test split
3. Converts all `.tif` files → `.npy` (10x faster loading during training)
4. Saves everything to Google Drive

**No GPU needed** — keep runtime as CPU to save your GPU quota.

**Expected time:** 20-40 minutes for 1200 images.

After this notebook finishes, the training notebook copies the `.npy` files
from Drive to Colab local disk and trains from there at full speed.

---
## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
def gb(x): return x / (1024**3)
total, used, free = shutil.disk_usage('/content')
print(f'Colab disk — total: {gb(total):.1f} GB | used: {gb(used):.1f} GB | free: {gb(free):.1f} GB')

---
## Cell 2 — Set Paths

In [ ]:
import os

# Dataset location on Drive
DATASET_PATH = '/content/drive/MyDrive/Geo_Spill_Data'

# Where results are saved on Drive
RESULTS_DIR = '/content/drive/MyDrive/Geo_Spill_results'
NPY_DIR     = f'{RESULTS_DIR}/npy_cache'

os.makedirs(f'{NPY_DIR}/images', exist_ok=True)
os.makedirs(f'{NPY_DIR}/masks',  exist_ok=True)

# Verify dataset exists
images = [f for f in os.listdir(f'{DATASET_PATH}/images') if f.endswith('.tif')]
masks  = [f for f in os.listdir(f'{DATASET_PATH}/masks')  if f.endswith('.tif')]
print(f'Dataset : {DATASET_PATH}')
print(f'Images  : {len(images)}')
print(f'Masks   : {len(masks)}')
print(f'NPY dir : {NPY_DIR}')

---
## Cell 3 — Clone GitHub Repository

In [ ]:
import os

GITHUB_URL = 'https://github.com/TigranBoyakhchyan/GeoSpill-AI'
REPO_NAME  = 'GeoSpill-AI'

if os.path.exists(f'/content/{REPO_NAME}'):
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

os.makedirs('data', exist_ok=True)
print(f'Working directory: {os.getcwd()}')

---
## Cell 4 — Install Dependencies

In [ ]:
!pip install -q rasterio
import rasterio
print(f'rasterio: {rasterio.__version__}')

---
## Cell 5 — Compute Stats and Save Split

Reads each `.tif` once from Drive to compute mean/std.
Saves `train_stats.json` and `splits.json` locally.
Expected time: 5-10 minutes.

In [ ]:
import sys
sys.path.insert(0, '/content/GeoSpill-AI')

import src.preprocess as pre

pre.IMAGES_DIR  = f'{DATASET_PATH}/images'
pre.MASKS_DIR   = f'{DATASET_PATH}/masks'
pre.STATS_FILE  = 'data/train_stats.json'
pre.SPLITS_FILE = 'data/splits.json'

pre.run_preprocessing()
print('Stats and splits saved.')

---
## Cell 6 — Convert TIF → NPY and Save to Drive

Each image is:
- Clipped to [-50, 0] dB
- Z-score normalized using the training stats
- Saved as `.npy` directly to Google Drive

This means at training time, files just need to be loaded and cropped — no processing.

**If this cell is interrupted**, just re-run it — it skips files that are already converted.

Expected time: **20-40 minutes** (Drive write speed is the bottleneck).

In [ ]:
import os, json
import numpy as np
import rasterio
from tqdm import tqdm
import warnings, logging
from rasterio.errors import NotGeoreferencedWarning
warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)
logging.getLogger('rasterio').setLevel(logging.ERROR)

# Load normalization stats
with open('data/train_stats.json') as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)[:, None, None]  # (2, 1, 1)
std  = np.array(stats['std'],  dtype=np.float32)[:, None, None]
print(f'Mean: {stats["mean"]}')
print(f'Std : {stats["std"]}')
print(f'Saving to: {NPY_DIR}\n')

def convert_image(src_path, dst_path):
    try:
        with rasterio.open(src_path) as src:
            img = src.read().astype(np.float32)   # (2, H, W)
        img = np.clip(img, -50.0, 0.0)
        img = (img - mean) / (std + 1e-6)
        np.save(dst_path, img)
        return True
    except Exception as e:
        print(f'\n  SKIPPING {os.path.basename(src_path)}: {e}')
        return False

def convert_mask(src_path, dst_path):
    try:
        with rasterio.open(src_path) as src:
            mask = src.read(1).astype(np.float32)  # (H, W)
        if mask.max() > 1.0:
            mask = (mask > 127).astype(np.float32)
        np.save(dst_path, mask)
        return True
    except Exception as e:
        print(f'\n  SKIPPING {os.path.basename(src_path)}: {e}')
        return False

# ── Convert images ────────────────────────────────────────────────────────
image_files = sorted([f for f in os.listdir(f'{DATASET_PATH}/images') if f.endswith('.tif')])
print(f'Converting {len(image_files)} images...')
img_ok = img_skip = img_exist = 0
for fname in tqdm(image_files):
    stem = os.path.splitext(fname)[0]
    dst  = f'{NPY_DIR}/images/{stem}.npy'
    if os.path.exists(dst):
        img_exist += 1
        continue
    if convert_image(f'{DATASET_PATH}/images/{fname}', dst):
        img_ok += 1
    else:
        img_skip += 1
print(f'Images — newly converted: {img_ok}, already existed: {img_exist}, failed: {img_skip}')

# ── Convert masks ─────────────────────────────────────────────────────────
mask_files = sorted([f for f in os.listdir(f'{DATASET_PATH}/masks') if f.endswith('.tif')])
print(f'\nConverting {len(mask_files)} masks...')
msk_ok = msk_skip = msk_exist = 0
for fname in tqdm(mask_files):
    stem = os.path.splitext(fname)[0]
    dst  = f'{NPY_DIR}/masks/{stem}.npy'
    if os.path.exists(dst):
        msk_exist += 1
        continue
    if convert_mask(f'{DATASET_PATH}/masks/{fname}', dst):
        msk_ok += 1
    else:
        msk_skip += 1
print(f'Masks — newly converted: {msk_ok}, already existed: {msk_exist}, failed: {msk_skip}')

# ── Summary ───────────────────────────────────────────────────────────────
n_img  = len([f for f in os.listdir(f'{NPY_DIR}/images') if f.endswith('.npy')])
n_mask = len([f for f in os.listdir(f'{NPY_DIR}/masks')  if f.endswith('.npy')])
print(f'\nTotal NPY on Drive — images: {n_img}, masks: {n_mask}')
print('Conversion complete!')

---
## Cell 7 — Update splits.json to Use NPY Paths

Rewrites splits so the training notebook loads `.npy` files instead of `.tif`.

In [ ]:
import json, os

with open('data/splits.json') as f:
    splits = json.load(f)

def to_npy_path(p, npy_dir, subfolder):
    stem = os.path.splitext(os.path.basename(p))[0]
    return f'{npy_dir}/{subfolder}/{stem}.npy'

splits['train'] = [to_npy_path(p, NPY_DIR, 'images') for p in splits['train']]
splits['val']   = [to_npy_path(p, NPY_DIR, 'images') for p in splits['val']]
splits['test']  = [to_npy_path(p, NPY_DIR, 'images') for p in splits['test']]
splits['masks'] = {
    to_npy_path(k, NPY_DIR, 'images'): to_npy_path(v, NPY_DIR, 'masks')
    for k, v in splits['masks'].items()
}

with open('data/splits.json', 'w') as f:
    json.dump(splits, f, indent=2)

print('splits.json updated to .npy paths')
print(f'Sample image : {splits["train"][0]}')
print(f'Sample mask  : {list(splits["masks"].values())[0]}')

---
## Cell 8 — Save Stats and Updated Splits to Drive

**Always run before closing the session.**

In [ ]:
import shutil, json

os.makedirs(RESULTS_DIR, exist_ok=True)
shutil.copy('data/train_stats.json', f'{RESULTS_DIR}/train_stats.json')
shutil.copy('data/splits.json',      f'{RESULTS_DIR}/splits.json')

with open('data/train_stats.json') as f: stats  = json.load(f)
with open('data/splits.json')      as f: splits = json.load(f)

print(f'Saved to: {RESULTS_DIR}')
print(f'  train_stats.json')
print(f'  splits.json — {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test')
print(f'  npy_cache/  — images and masks as .npy')
print('\nDone! Run the training notebook next.')